In [1]:

import pandas as pd
from pathlib import Path

# Locate the project folder
project_root = Path.cwd()

if not (project_root / "Data").exists():
    project_root = project_root.parent

# Load original dataset
file_path = project_root / "Data" / "raw" / "Online Retail.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

print("Original rows:", len(df))


Original rows: 541909


In [2]:

print("Exact duplicate rows:", df.duplicated().sum())

# Show examples of duplicate records
duplicate_examples = df[
    df.duplicated(keep=False)
].sort_values(["InvoiceNo", "StockCode"])

display(duplicate_examples.head(10))


Exact duplicate rows: 5268


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [3]:

sales_df = df.copy()

# Keep positive sales transactions
sales_df = sales_df[
    (sales_df["Quantity"] > 0) &
    (sales_df["UnitPrice"] > 0) &
    (sales_df["Description"].notna()) &
    (~sales_df["InvoiceNo"].astype(str).str.startswith("C"))
].copy()

# Calculate revenue for each transaction line
sales_df["Revenue"] = (
    sales_df["Quantity"] * sales_df["UnitPrice"]
)

# Extract month for reporting
sales_df["YearMonth"] = (
    sales_df["InvoiceDate"].dt.strftime("%Y-%m")
)

print("Sales rows:", len(sales_df))
print("Gross positive sales revenue:",
      round(sales_df["Revenue"].sum(), 2))

display(sales_df.head())


Sales rows: 530104
Gross positive sales revenue: 10666684.54


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


In [4]:

customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

# Convert customer IDs to strings
customer_df["CustomerID"] = (
    customer_df["CustomerID"].astype(int).astype(str)
)

print("Customer transaction rows:", len(customer_df))
print("Unique customers:",
      customer_df["CustomerID"].nunique())

display(customer_df.head())


Customer transaction rows: 397884
Unique customers: 4338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


In [5]:

output_folder = project_root / "Data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

sales_df.to_csv(
    output_folder / "sales_transactions.csv",
    index=False
)

customer_df.to_csv(
    output_folder / "customer_transactions.csv",
    index=False
)

print("Both datasets saved successfully!")


Both datasets saved successfully!
